# 05 · Orchestration

So far the agent decided every step. Many systems are better as a **workflow**, where **code** decides
the steps and the model fills in the parts that need language. CrewAI's answer is the **Flow**: plain
Python methods, wired together with decorators, sharing a typed state. Agents and crews run inside its
steps.

**You will learn**

- how to build a Flow: `@start`, `@listen`, `@router` and the flow's state
- when to use a workflow, an agent, or an agent inside a workflow
- how to build a loop whose limit is enforced in code, not in the prompt
- the research agent's design, which the capstone builds on

In [1]:
from crew_agents.llm import make_llm  # import crew_agents first: it switches CrewAI's telemetry off
from crew_agents.prompts import BACKSTORY, GOAL, ROLE
from crew_agents.tools import shipfast_tools
from crewai import Agent
from crewai.flow.flow import Flow, listen, router, start
from pydantic import BaseModel

from agent_contracts.harness import verify_model
from agent_contracts.shipfast import tools as shipfast
from agent_contracts.shipfast.schema import Triage

verify_model()
llm = make_llm()

## A first flow: routing ShipFast messages

A Flow is a class. Its **state** is a Pydantic model every step can read and change (`self.state`).
`@start()` marks the first step; `@listen(step)` runs a method when another finishes; and `@router(step)`
runs after a step and returns a **label**, so only the listeners for that label run next.

ShipFast's first decision is a good fit for code: triage the message (notebook 01), and if it is out of
scope, decline without ever starting the specialist agent.

In [2]:
TRIAGE_PROMPT = ("Classify the customer's message for ShipFast support. intent: order_status, refund_request, "
                 "complaint, or other (anything not about orders, deliveries, refunds or complaints). "
                 "order_id: the order ID such as A1001 if given, else null.")
specialist = Agent(role=ROLE, goal=GOAL, backstory=BACKSTORY, tools=shipfast_tools(), llm=llm, verbose=False)


class Support(BaseModel):
    message: str = ""
    intent: str = ""
    order_id: str | None = None
    reply: str = ""


class SupportFlow(Flow[Support]):
    @start()
    def triage(self):
        t = llm.call([{"role": "system", "content": TRIAGE_PROMPT}, {"role": "user", "content": self.state.message}],
                     response_model=Triage)
        self.state.intent, self.state.order_id = t.intent, t.order_id

    @router(triage)
    def route(self):
        return "decline" if self.state.intent == "other" else "resolve"

    @listen("decline")
    def send_decline(self):  # no model call at all
        self.state.reply = "Sorry, I can only help with orders, deliveries, refunds and complaints."

    @listen("resolve")
    async def run_specialist(self):
        self.state.reply = (await specialist.kickoff_async(self.state.message)).raw

In [3]:
for message in ("Can you write me a poem about cats?", "Where is order A9999?"):
    shipfast.reset()
    flow = SupportFlow(suppress_flow_events=True)  # no console panel for every step
    await flow.kickoff_async(inputs={"message": message})
    s = flow.state
    print(f"{message}\n  triage: {s.intent}, {s.order_id}   tools: {shipfast.tool_calls()}")
    print(f"  reply:  {s.reply.strip()[:150]}\n")

Can you write me a poem about cats?
  triage: other, None   tools: []
  reply:  Sorry, I can only help with orders, deliveries, refunds and complaints.



Where is order A9999?
  triage: order_status, A9999   tools: ['lookup_order']
  reply:  I couldn't find any order with the ID A9999. Could you please double-check the order ID or provide additional details?



The poem never reached the specialist: a router in code handled it, cheaply and predictably. The order
question went to the agent, which chose its own tools. That mix, **a workflow outside and an agent
inside**, is what CrewAI recommends: start with a Flow for structure, and use agents or crews inside the
steps that need autonomy.

Three practical notes. `suppress_flow_events=True` stops CrewAI printing a panel for every step of the
flow. A listener cannot share its name with the label it listens to: CrewAI rejects
`@listen("decline") def decline` as a possible infinite loop, so the handlers here are `send_decline` and
`run_specialist`. And `kickoff_async` again, because a notebook already runs an event loop; the
`run_specialist` step is `async` for the same reason, since it awaits the agent.

## The research agent as a flow

The research agent (`contracts/agent_contracts/research/SPEC.md`) must plan, research with at most 5 tool
calls, draft a cited answer, critique it, and revise at most twice. Most of that is a fixed sequence, so it
is a flow. Only the research step is open-ended, so that step is an agent. It lives in
`crew_agents/research_flow.py`, with the same instructions as the LangGraph build:

In [4]:
import crew_agents.research_flow as rf

print(rf.__doc__)

The research agent as a CrewAI Flow (research SPEC.md section 1), taught in notebook 05.

Workflow outside, agent inside, as in lg_agents/research_flow.py and with the same instructions: fixed
steps (plan, draft, critique, revise) and a tool-using CrewAI agent for the research step. The revision
limit is enforced in code by the router, never by the prompt; the tool budget (5) by the shared tools.

    plan -> research -> draft -> critique (router) --"done"--> END
                                    ^        \--"needs_revision"--> revise --+
                                    +---------------------------------------+
    The router skips the critique and returns "done" once 2 revisions have run.



## Running it

Question Q01 asks how many employees Veltara Aquatech has. The fixtures contain the current figure (350,
in a 2026 article) and a stale one (200, in a 2023 article), and an employees search returns both.

In [5]:
import time

from agent_contracts.research import tools as research
from agent_contracts.research.scoring import load_questions, score_report

questions = {q["id"]: q for q in load_questions()}
q = questions["Q01_headcount"]
research.reset()
flow = rf.ResearchFlow(llm)
start_time = time.perf_counter()
await flow.kickoff_async(inputs={"question_id": q["id"], "question": q["question"]})
state = flow.state
print(f"ran in {time.perf_counter() - start_time:.0f}s")
print("plan:     ", state.plan)
print("tools:    ", research.tool_calls())
print("answer:   ", state.answer)
print("citations:", state.citations)
print("critiques:", [c["verdict"] for c in state.critiques], "| revisions:", state.revisions)
print("score:    ", score_report(rf.to_report(state), q))

ran in 52s
plan:      ["Search for Veltara Aquatech's official website to find any publicly available information about their employee count.", 'Look for recent press releases, investor reports, or company profiles on platforms like LinkedIn or Crunchbase that may mention the number of employees.', 'Check if the company has disclosed its employee count in annual reports or through regulatory filings (e.g., SEC filings for U.S.-based companies).', 'If the information is not publicly available, consider reaching out to the company through their contact form or customer service for an official response.']
tools:     ['web_search']
answer:    Veltara Aquatech currently has 350 employees, according to the most recent report published on 2026-03-18 [F02].
citations: ['F02']
critiques: ['pass'] | revisions: 0
score:     {'passed': True, 'scores': {'fact_recall': 1.0, 'facts_missing': [], 'contradiction_free': True, 'citations_valid': True}, 'problems': []}


When we ran it, the flow answered with the current figure, 350, cited the 2026 article, and the critic
passed it at the first attempt, so the loop never ran. The plan is generic (it suggests LinkedIn,
regulatory filings and emailing the company, none of which the tools can do); it guides the research
agent but nothing checks it.

The LangGraph tutorial ran the same question with the same instructions, and there the draft added that
350 "supersedes" the older 200. The critic passed that too, but the shared scorer failed it for mentioning
the stale figure. A model checking a model misses things a plain text check does not, which is why both
agents are scored in code.

## The loop, and its limit

The loop is two decorators. `critique` is a router that runs after `draft` **or** after `revise`; it
returns `"needs_revision"`, which triggers `revise`, or `"done"`, which nothing listens to, so the flow
ends. The limit sits at the top of the router, in code:

In [6]:
import inspect

print(inspect.getsource(rf.ResearchFlow.critique))

    @router(or_(draft, "revise"))  # "revise" here is the method below finishing
    def critique(self):
        if self.state.revisions >= MAX_REVISIONS:  # the limit, enforced in code: no third critique
            return "done"
        c = critique_draft(self.llm, self.state.question, self.state.findings, self.state.answer, self.state.citations)
        self.state.critiques.append(c.model_dump())
        return "needs_revision" if c.verdict == "revise" else "done"



The critic is not useless: given a draft that relies on the stale figure outright, with the real
findings from the run above, it sends it back:

In [7]:
print(rf.critique_draft(llm, q["question"], state.findings, "Veltara Aquatech has 200 employees [F03].", ["F03"]))

verdict='revise' issues=['The draft answer cites F03, which reports 200 employees, but the most recent finding (F02) from 2026-03-18 states that Veltara Aquatech now has 350 employees. The draft answer uses outdated information.', 'The question asks for the current number of employees, and the draft answer does not provide the most recent figure as per the findings.']


To see the limit hold, replace the critic with one that is never satisfied, a standard testing trick,
and run another question. However often the critic says "revise", the flow stops after the second
revision without asking a third time.

In [8]:
from agent_contracts.research.schema import Critique

real_critic = rf.critique_draft
critic_calls = []


def never_satisfied(*args, **kwargs):
    critic_calls.append(1)
    return Critique(verdict="revise", issues=["(test) always revise"])


rf.critique_draft = never_satisfied
try:
    q4 = questions["Q04_series_b"]
    research.reset()
    flow = rf.ResearchFlow(llm)
    await flow.kickoff_async(inputs={"question_id": q4["id"], "question": q4["question"]})
finally:
    rf.critique_draft = real_critic

report = rf.to_report(flow.state)
print("critic calls:", len(critic_calls), "| verdicts:", [c.verdict for c in report.critiques], "| revisions:", report.revisions)
print("loop shape problems:", [p for p in score_report(report, q4)["problems"] if "loop" in p] or "none")

critic calls: 2 | verdicts: ['revise', 'revise'] | revisions: 2
loop shape problems: none


So a run can only end in the three shapes the spec allows: `[pass]`, `[revise, pass]`, or
`[revise, revise]` after two revisions. `score_report` checks that shape for every report.

## Workflow or agent?

| | Workflow (`Flow`) | Agent (`Agent`, or a `Crew` of agents) |
|---|---|---|
| Who decides the next step | your code | the model |
| Predictability | high: the same path every time | lower: it can skip or repeat steps |
| Good for | known procedures, limits that must hold, cheap routing | open-ended investigation |
| In this tutorial | ShipFast routing, the research loop | ShipFast's specialist, the research step |

A `Crew` is CrewAI's other way to combine work: several agents and tasks run by a **process**, either
`sequential` (tasks in order) or `hierarchical` (a manager agent plans and delegates). That is
orchestration by agents rather than by code; Flows are the code-first option, and the two combine.

## How the others do this

- **LangGraph** draws the same thing as a `StateGraph`: a typed state, nodes, and conditional edges whose
  routing functions return the next node's name, the counterpart of a CrewAI router's label.
- **Strands** builds multi-step systems from agents: `GraphBuilder` connects agents as nodes with
  conditional edges, and `Swarm` lets agents hand off to each other. Plain code steps are less central
  than in a Flow or a `StateGraph`.

## Takeaways

- A Flow is a class: `@start`, `@listen` and `@router` wire methods together, and `self.state` carries the
  data between them.
- Use code for the steps you know and an agent for the steps you do not; mix them freely.
- Put limits in the routers. A loop bounded in code cannot run away.
- Next: **06 · Guardrails and approval**.